# 01 Data Analysis And Prep
Этапы 1-5: загрузка источника, базовая диагностика, подготовка и сохранение промежуточного датасета на диск.

In [1]:
import pandas as pd

from _shared_notebook_utils import DATA_DIR, ROOT, ensure_dirs, save_json, save_pickle

ensure_dirs()
print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)

ROOT = C:\Users\Dmitry\code-projects\diploma-crop-rotation
DATA_DIR = C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data


## 1) Загрузка исходных данных
По умолчанию используется локальный кэш national1724.pkl; при необходимости можно заменить на чтение из GDB.

In [ ]:
candidate_cache_paths = [
    ROOT / 'national1724.pkl',
    ROOT / 'data' / 'national1724.pkl',
    ROOT / 'datasets' / 'national1724.pkl',
]
cache_path = next((p for p in candidate_cache_paths if p.exists()), None)
if cache_path is None:
    searched = '\n'.join(str(p) for p in candidate_cache_paths)
    raise FileNotFoundError(
        'Не найден исходный датасет. Поместите national1724.pkl в директорию проекта.\n'
        f'Проверены пути:\n{searched}'
    )

df_raw = pd.read_pickle(cache_path)
print('cache_path =', cache_path)
print('df_raw shape:', df_raw.shape)
display(df_raw.head(3))

df_raw shape: (16418258, 20)


,CSBID,CSBYEARS,CSBACRES,CDL2017,CDL2018,CDL2019,CDL2020,CDL2021,CDL2022,CDL2023,CDL2024,STATEFIPS,STATEASD,ASD,CNTY,CNTYFIPS,INSIDE_X,INSIDE_Y,Shape_Length,Shape_Area
0,351724000000001,1724,2.777993,24,24,152,152,152,152,152,152,35,3530,30,Union,059,-650336.7078,1.447441e+06,628.315309,11242.149046
1,351724000000002,1724,9.293377,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-648203.5992,1.447142e+06,1042.818836,37608.996229
2,351724000000003,1724,5.775892,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-647571.4914,1.447064e+06,1164.537470,23374.226788


## 2) Первичная диагностика
Фиксируем список CDL-колонок и базовые показатели качества данных.

In [3]:
cdl_cols = [c for c in df_raw.columns if c.startswith('CDL') and c[3:7].isdigit()]
id_cols = [c for c in ['CSBID', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y'] if c in df_raw.columns]

diag_df = pd.DataFrame({
    'column': id_cols + cdl_cols[:10],
    'dtype': [str(df_raw[c].dtype) for c in id_cols + cdl_cols[:10]],
    'missing': [int(df_raw[c].isna().sum()) for c in id_cols + cdl_cols[:10]]
})

print('CDL columns found:', len(cdl_cols))
display(diag_df)

CDL columns found: 8


,column,dtype,missing
0,CSBID,str,0
1,CSBACRES,float64,0
2,STATEFIPS,str,0
3,CNTYFIPS,str,0
4,INSIDE_X,float64,0
5,INSIDE_Y,float64,0
6,CDL2017,int32,0
7,CDL2018,int32,0
8,CDL2019,int32,0
9,CDL2020,int32,0


## 3) Подготовка и очистка (migration placeholder)
На этом шаге переносим рабочую логику из scripts.ipynb: маппинг CDL в группы, добавление *_name/*_group, удаление bad/unknown классов.

В текущей версии временно используем упрощенный вариант без legacy fallback: берется копия исходного df_raw до полного переноса prep-логики.

In [ ]:
# Temporary placeholder until full prep logic is migrated from scripts.ipynb.
df_prepared_filtered = df_raw.copy()
prep_source = 'raw_copy_temporary_no_legacy_fallback'

print('prep_source =', prep_source)
print('df_prepared_filtered shape:', df_prepared_filtered.shape)

prep_source = fallback_checkpoint_window_df_filtered
df_prepared_filtered shape: (38511210, 11)


## 4) Сохранение артефактов этапа
Сохраняем DataFrame и metadata для downstream ноутбуков.

In [5]:
prepared_path = DATA_DIR / '01_df_prepared_filtered.pkl'
size_mb = save_pickle(df_prepared_filtered, prepared_path)

meta = {
    'source_cache': str(cache_path),
    'prep_source': prep_source,
    'prepared_path': str(prepared_path),
    'rows': int(df_prepared_filtered.shape[0]),
    'cols': int(df_prepared_filtered.shape[1]),
    'cdl_cols_count': int(len(cdl_cols))
}

meta_path = DATA_DIR / '01_meta.json'
save_json(meta, meta_path)
print('Saved:', prepared_path, f'({size_mb:.2f} MB)')
print('Saved:', meta_path)

Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\01_df_prepared_filtered.pkl (4263.56 MB)
Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\01_meta.json
